# Template 09d: State Lift

Value-adjusted decile lift per state (`st_raw` from the master file). Small states have few claims - check `n_claims` and `bins` in the summary before reading their charts.

**Value-adjusted lift** (documentation/ModelMath_v2.md): the loss was divided by the depreciation
factor D upstream and the GLM carries D, so D is multiplied back in:
- Actual `A = (PP / GLM) x D`, Predicted `P = (PRED / GLM) x D`
- Rows sorted by P, exposure-weighted groups (equal exposure per group)
- Relativity = group mean / overall predicted mean (of that data); A/P = actual / predicted

`test_holdout` = test (05c) + holdout (08) stacked (disjoint folds). `train` is shown for comparison:
the model saw those losses, so train lift is optimistic.

Every chart PNG has a CSV with the same name next to it in `results/09/`.
CSV columns are described in documentation/ModelMath_v2.md, section "Stage 09 Lift Tables".

One full-size decile chart per state, largest exposure first. Groups with exposure below
`value_lift.min_group_exposure` use 5 groups instead of deciles (see `bins` in the summary).

**Outputs (results/09/):**
- 09d_summary.csv (per state and dataset: exposure, claims, A/P, gini, top/bottom lift)
- 09d_lift_{dataset}.csv (all state decile tables in one file, for filtering)
- 09d_{dataset}/09d_{rank}_{state}.png + .csv (one chart and its table per state)

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import os
import sys
import yaml
import pandas as pd
import matplotlib
matplotlib.use("Agg")
from pathlib import Path
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment
project_root = setup_notebook_environment()
import lift_analysis as la

pd.set_option("display.width", 200)
print("########################################")
print("# STAGE 09d: STATE LIFT")
print("########################################")

In [ ]:
pc_id = f"PC{open('current.pc').read().strip()}"
with open(f"{config_path}/config.yaml") as f:
    cfg = yaml.safe_load(f)
output_base = cfg["machines"][pc_id]["paths"]["output_path"]
model_name = cfg["experiment"]["name"]
vl = cfg.get("value_lift", {})
value_adjust = vl.get("value_adjust", True)
datasets = vl.get("datasets", ["test_holdout", "train"])
segment_file = vl.get("segment_file", "config/company_segment.csv")
min_group_exposure = vl.get("min_group_exposure", 5000)
out_dir = la.stage_dir(output_base)  # results/09: chart PNG + table CSV side by side
label = "value-adjusted" if value_adjust else "ratio (no Dep_factor)"
print(f"Model: {model_name} | output: {out_dir}")
print(f"Datasets: {datasets} | value_adjust: {value_adjust}")

In [ ]:
summaries, charts = [], {}
for ds in datasets:
    df = la.lift_frame(output_base, cfg, pc_id, ds, value_adjust=value_adjust, groups=('state',))
    if df is None:
        print(f"[WARN] {ds}: no data, skipped")
        continue
    print(f"\n=== {ds}: {len(df):,} rows, {df['state'].nunique()} states ===")
    tables, s = la.group_lift(df, "state", bins=10, min_exposure=min_group_exposure)
    la.stack_tables(tables, "state").to_csv(f"{out_dir}/09d_lift_{ds}.csv", index=False)
    group_dir = la.reset_dir(f"{out_dir}/09d_{ds}")
    charts[ds] = la.save_group_lifts(tables, s, "state", f"{model_name} - {ds}: {label} decile lift",
                                     group_dir, "09d")
    print(f"  saved {len(charts[ds])} charts (PNG + CSV) to {group_dir}")
    s.insert(0, "dataset", ds)
    summaries.append(s)
    del df

summary = pd.concat(summaries, ignore_index=True)
summary.insert(1, "value_adjust", value_adjust)
summary.to_csv(f"{out_dir}/09d_summary.csv", index=False)
cols = ["dataset", "state", "exposure_pct", "n_records", "n_claims", "bins", "a_to_p", "gini",
        "lift_top_bottom_actual", "lift_top_bottom_pred", "mean_abs_a_to_p_error"]
summary[cols].round(3)

## Train vs test+holdout

In [ ]:
# A state whose train gini is far above its test_holdout gini has unreliable lift
if summary["dataset"].nunique() > 1:
    cmp = summary.pivot_table(index="state", columns="dataset", values=["exposure_pct", "a_to_p", "gini"])
    cmp.columns = [f"{v}_{d}" for v, d in cmp.columns]
    cmp = cmp.sort_values(cmp.columns[0], ascending=False)
    display(cmp.round(3))

## Lift charts: test + holdout

In [ ]:
for g, path in charts.get("test_holdout", {}).items():
    display(Image(filename=path))

## Lift charts: train

In [ ]:
for g, path in charts.get("train", {}).items():
    display(Image(filename=path))

In [ ]:
print("\n########################################")
print("# STAGE 09d: COMPLETE")
print("########################################")